# Sauda v2 — DPO on top of GRPO (Colab T4)

Third training stage. Takes the v2 SFT+GRPO buyer (`PayMyBills/bestdealbot-v2`)
and does Direct Preference Optimization on (chosen, rejected) pairs generated
by Claude as judge.

**Pipeline:**
1. Pull v2 base + adapter from HF
2. Load DPO pairs from `data/dpo_pairs.jsonl` (built offline by `eval/build_dpo_pairs.py`)
3. Run `trl.DPOTrainer` for 1 epoch
4. Push merged adapter to HF as `PayMyBills/bestdealbot-v3-dpo`

**Setup:** Runtime → T4 GPU; sidebar 🔑 → `HF_TOKEN`.


## 1 · Install

In [ ]:
!pip install -q --upgrade "trl>=0.12" "peft>=0.13" "transformers>=4.46" \
    "accelerate>=1.1" "bitsandbytes>=0.44" "datasets>=3.0" "huggingface_hub>=0.27"


## 2 · HF auth

In [ ]:
import os
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)
print("auth OK")


## 3 · Clone repo + load DPO pairs

In [ ]:
import os, shutil, sys, subprocess, json
REPO_URL = "https://github.com/paymybills/BazaarBATNA.git"
LOCAL = "/content/metathon"
if os.path.exists(LOCAL):
    shutil.rmtree(LOCAL)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, LOCAL], check=True)
sys.path.insert(0, LOCAL)
os.chdir(LOCAL)

# DPO pairs should live at data/dpo_pairs.jsonl (built offline). If missing,
# upload the file via Colab Files panel (left sidebar → folder icon → upload).
PAIRS_PATH = "data/dpo_pairs.jsonl"
if not os.path.exists(PAIRS_PATH):
    raise FileNotFoundError(
        f"{PAIRS_PATH} missing. Build it locally with `eval/build_dpo_pairs.py` "
        "and upload to Colab, or run that script in this Colab session first."
    )

pairs = [json.loads(l) for l in open(PAIRS_PATH)]
print(f"Loaded {len(pairs)} DPO pairs")
print("Sample keys:", list(pairs[0].keys()))


## 4 · Load v2 base + adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model

BASE_MODEL = "unsloth/Llama-3.1-8B-Instruct"
V2_ADAPTER = "PayMyBills/bestdealbot-v2"  # SFT+GRPO LoRA
V3_REPO = "PayMyBills/bestdealbot-v3-dpo"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)

# Layer the v2 adapter on top
model = PeftModel.from_pretrained(model, V2_ADAPTER, is_trainable=True)
model.print_trainable_parameters()


## 5 · Build DPO dataset

In [ ]:
from datasets import Dataset

# trl.DPOTrainer expects {prompt, chosen, rejected}
def to_dpo_row(pair):
    # The "prompt" we wrote during pair-building is a JSON blob; for DPO
    # we want a chat-template-rendered prompt. Use a simple conversation prefix.
    return {
        "prompt": f"You are a buyer negotiating. Listing: {pair['listing_title']}. Round 1.",
        "chosen": pair["chosen"],
        "rejected": pair["rejected"],
    }

dpo_ds = Dataset.from_list([to_dpo_row(p) for p in pairs])
print(f"DPO dataset: {len(dpo_ds)} rows")
print(dpo_ds[0])


## 6 · DPO training

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_cfg = DPOConfig(
    output_dir="/content/bestdealbot-v3-dpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=2,
    save_strategy="no",
    bf16=True,
    report_to="none",
    beta=0.1,  # DPO temperature — lower = more conservative
    max_length=1024,
    max_prompt_length=512,
    push_to_hub=True,
    hub_model_id=V3_REPO,
    hub_strategy="end",
    hub_private_repo=False,
)
# SFTTrainer-style checkpointing for training stability
model.gradient_checkpointing_enable()
model.config.use_cache = False

trainer = DPOTrainer(
    model=model,
    args=dpo_cfg,
    processing_class=tokenizer,
    train_dataset=dpo_ds,
)
trainer.train()
print("DPO training complete.")


## 7 · Push final adapter

In [ ]:
model.save_pretrained("/content/bestdealbot-v3-dpo")
tokenizer.save_pretrained("/content/bestdealbot-v3-dpo")
from huggingface_hub import HfApi
HfApi().upload_folder(
    folder_path="/content/bestdealbot-v3-dpo",
    repo_id=V3_REPO,
    repo_type="model",
    commit_message="DPO adapter on top of v2 SFT+GRPO",
)
print(f"Pushed to https://huggingface.co/{V3_REPO}")
